<a id="top"></a>

# L12 lecture: Non-determinism, pass rates, and a model A/B

```yaml
title: "L12 lecture: Non-determinism, pass rates, and a model A/B"
keywords: evaluation, non-determinism, pass rate, samples, model comparison, sonnet, haiku, regression, measurement instrument, teacher lecture
estimated duration: 30
```

> **Lesson:** L12 Evaluation. Roadmap: [objectives.md](../../../../docs/origin/lesson_roadmaps/L12/objectives.md) (objective 3).
>
> Runs offline — no API key needed. The model contrast is **simulated deterministically** so the page reproduces; an optional, clearly-marked cell runs the *real* Sonnet-vs-Haiku A/B if a key is present.

## Contents

- [1. Setup — two simulated models](#1-setup--two-simulated-models)
- [2. One run can be luck](#2-one-run-can-be-luck)
- [3. Measure rates, not verdicts](#3-measure-rates-not-verdicts)
- [4. Same eval set, two models](#4-same-eval-set-two-models)
  - [4.1 (Optional, live) the real Sonnet-vs-Haiku A/B](#41-optional-live-the-real-sonnet-vs-haiku-ab)
- [5. The same machinery is a regression check](#5-the-same-machinery-is-a-regression-check)
- [6. Takeaways](#6-takeaways)

## 1. Setup — two simulated models

L11 told you the loop is **non-deterministic**: the same task can take a different path each run, so *a single differing run proves nothing*. L12 builds the disciplined version of that warning.

To show it without a live model (and without flakiness we can't reproduce on the page), we **simulate** two models with a scripted `FakeModel`:

- a **strong** model that chains correctly and recovers from a tool error;
- a **weak** model that gives up on recovery and *intermittently runs away* on the lookup task.

`ModelSim` is keyed by `(case.id, attempt#)`, so it's flaky in a reproducible way. The two scorers — `answer_correct` (loosened to ignore commas) and `no_runaway` — apply to every case.

In [1]:
from fluffy_potato_curriculum.common.agent_loop import RunResult, run
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    compare,
    evaluate,
    tool_calls,
)
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS


def _norm(text: str) -> str:
    """Drop commas/spaces so '37,000,000' and '37000000' compare equal."""
    return text.replace(",", "").replace(" ", "")


def answer_correct(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer (loosened): is the reference answer in the final text?"""
    expected = str(example.reference_outputs["answer"])
    return EvalResult(key="answer_correct", score=_norm(expected) in _norm(run.final_text))


def no_runaway(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: no repeated (tool, args) call and didn't hit the step cap."""
    signatures = [(name, tuple(sorted(args.items()))) for name, args in tool_calls(run)]
    repeated = len(signatures) != len(set(signatures))
    return EvalResult(key="no_runaway", score=not (repeated or run.termination == "max_steps"))


class ModelSim:
    """A run_case that simulates a model of a given strength, deterministically.

    Strong: recovers from the crash (retries https://ok) and looks Tokyo up once.
    Weak: gives up on recovery, and on every third lookup attempt runs away on a
    bad city until max_steps. Behavior is keyed by (case.id, attempt#).
    """

    def __init__(self, *, strong: bool) -> None:
        self.strong = strong
        self._seen: dict[str, int] = {}

    def _attempt(self, case_id: str) -> int:
        n = self._seen.get(case_id, 0)
        self._seen[case_id] = n + 1
        return n

    def __call__(self, case: EvalCase) -> RunResult:
        n = self._attempt(case.id)
        if case.id == "recover" and self.strong:
            script = [
                tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
                tool_reply(tool_call("c2", "flaky_fetch", {"url": "https://ok"})),
                text_reply("The answer is 42."),
            ]
        elif case.id == "recover":
            script = [
                tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
                text_reply("Sorry, I could not fetch that URL."),
            ]
        elif case.id == "hard_lookup" and not self.strong and n % 3 == 1:
            # the weak model flakes into a runaway on every third attempt.
            script = [tool_reply(tool_call("c1", "lookup", {"city": "Atlantis"}))]
        elif case.id == "hard_lookup":
            script = [
                tool_reply(tool_call("c1", "lookup", {"city": "Tokyo"})),
                text_reply("Tokyo has a population of 37000000."),
            ]
        else:
            raise KeyError(case.id)
        return run(
            FakeModel(script), TOOLS, case.inputs["task"], max_steps=case.inputs.get("max_steps", 8)
        )


recover_case = EvalCase(
    id="recover",
    inputs={"task": "Fetch the answer from https://crash; if it fails, try https://ok."},
    reference_outputs={"answer": "42"},
)
hard_lookup_case = EvalCase(
    id="hard_lookup",
    inputs={"task": "What is the population of Tokyo?", "max_steps": 4},
    reference_outputs={"answer": "37000000"},
)
ab_cases = [recover_case, hard_lookup_case]
scorers = [answer_correct, no_runaway]
print("setup ready:", [c.id for c in ab_cases])

setup ready: ['recover', 'hard_lookup']


[↑ Back to top](#top)

## 2. One run can be luck

Run the weak model on the lookup case **once** (`samples=1`). It passes — looks fine. Run it again. It fails. *Which run do we believe?*

This confusion **is** the lesson. A single pass on a non-deterministic agent can be luck (and a single fail can be an unlucky path).

In [2]:
weak_once = ModelSim(strong=False)

print("first run (samples=1):")
print(evaluate(weak_once, [hard_lookup_case], scorers).table())

print("\nsecond run (samples=1):")
print(evaluate(weak_once, [hard_lookup_case], scorers).table())

first run (samples=1):
case         answer_correct  no_runaway  ALL
hard_lookup  1/1             1/1         1/1

second run (samples=1):
case         answer_correct  no_runaway  ALL
hard_lookup  0/1             0/1         0/1


First run green, second run red — same case, same model, no code changed. Believing either single run is believing a coin flip.

[↑ Back to top](#top)

## 3. Measure rates, not verdicts

The cheapest honest fix: run each case **K times** and report how often it passes — a **pass rate**, not a single verdict. `evaluate(..., samples=K)` does exactly this. With `K=3` the flaky case reads as `2/3`: a *finding*, not noise to ignore. (A flaky case needs a looser check, more samples, or a redesign — not a shrug.)

In [3]:
weak = ModelSim(strong=False)
report_k = evaluate(weak, [hard_lookup_case], scorers, samples=3)
print(report_k.table())
print("\nno_runaway pass rate:", report_k.pass_rate("hard_lookup", "no_runaway"))

case         answer_correct  no_runaway  ALL
hard_lookup  2/3             2/3         2/3

no_runaway pass rate: 0.6666666666666666


[↑ Back to top](#top)

## 4. Same eval set, two models

Now the headline. Run the **same** eval set against the strong model and the weak model, `K=3` samples each, and put the pass-rate tables side by side. Watch the weak model's rates **drop** — especially on the recovery case and the flaky multi-step lookup.

This turns "the weaker model is worse here" from an assertion into a **number**. The eval set is not just a regression guard; it's a **measurement instrument**.

In [4]:
strong_report = evaluate(ModelSim(strong=True), ab_cases, scorers, samples=3)
weak_report = evaluate(ModelSim(strong=False), ab_cases, scorers, samples=3)

print("=== strong model (Sonnet stand-in) ===")
print(strong_report.table())
print("\n=== weak model (Haiku stand-in) ===")
print(weak_report.table())

=== strong model (Sonnet stand-in) ===
case         answer_correct  no_runaway  ALL
recover      3/3             3/3         3/3
hard_lookup  3/3             3/3         3/3

=== weak model (Haiku stand-in) ===
case         answer_correct  no_runaway  ALL
recover      0/3             3/3         0/3
hard_lookup  2/3             2/3         2/3


The strong model is green across the board; the weak model drops to `0/3` on recovery (it gives up instead of retrying) and `2/3` on the lookup (it intermittently runs away). A green eval set on the strong model would have told you *nothing* about this headroom — you only see it by running the *same cases* against both.

### 4.1 (Optional, live) the real Sonnet-vs-Haiku A/B

The cell below runs the **actual** A/B against Claude **Sonnet 4.6** and **Haiku 4.5** through the loop. It is **gated on `ANTHROPIC_API_KEY`** (read via the `common/config` seam) — with no key set it prints a note and skips, so a keyless restart-and-run-all still passes. Live results vary run to run; that variance is the whole point of the pass rate.

In [5]:
from fluffy_potato_curriculum.common.config import get_settings

live_case = EvalCase(
    id="hard_lookup",
    inputs={"task": "What is the population of Tokyo? Use the lookup tool.", "max_steps": 5},
    reference_outputs={"answer": "37000000"},
)


def live_run_case(model_id: str):
    """A run_case that drives the loop with a real LangChain chat model for one model."""
    from langchain.chat_models import init_chat_model

    # The key loads through the config seam (a repo-root .env), not os.environ, so
    # pass it explicitly -- init_chat_model would otherwise not find it.
    model = init_chat_model(model_id, api_key=get_settings().anthropic_api_key)

    def run_case(case: EvalCase) -> RunResult:
        return run(model, TOOLS, case.inputs["task"], max_steps=case.inputs.get("max_steps", 8))

    return run_case


if get_settings().anthropic_api_key:
    for label, model_id in [
        ("Sonnet 4.6", "anthropic:claude-sonnet-4-6"),
        ("Haiku 4.5", "anthropic:claude-haiku-4-5-20251001"),
    ]:
        report = evaluate(live_run_case(model_id), [live_case], scorers, samples=3)
        print(f"=== {label} ({model_id}) ===")
        print(report.table())
        print()
else:
    print("No ANTHROPIC_API_KEY set — skipping the live A/B.")
    print("Set it in your environment or repo-root .env to run Sonnet vs Haiku for real.")

No ANTHROPIC_API_KEY set — skipping the live A/B.
Set it in your environment or repo-root .env to run Sonnet vs Haiku for real.


[↑ Back to top](#top)

## 5. The same machinery is a regression check

Reframe the *exact same* comparison as a regression check. Treat the strong model's report as the **baseline** and ask: which (case, scorer) pairs went **pass → fail**? `compare(before, after)` answers it — a pair "passes" when its rate hits `min_rate` (default `1.0`: green on every sample).

The same call answers "did my prompt edit break anything that used to work?" — *same machinery, different question.*

In [6]:
delta = compare(strong_report, weak_report)
print("regressions (pass -> fail):", delta.regressions)
print("fixes       (fail -> pass):", delta.fixes)
print("has regressions:", delta.has_regressions)

regressions (pass -> fail): [('recover', 'answer_correct'), ('hard_lookup', 'answer_correct'), ('hard_lookup', 'no_runaway')]
fixes       (fail -> pass): []
has regressions: True


Swapping in the weaker model regressed recovery and the lookup — caught *before* shipping, by re-running cases you already had. That is the **ratchet**: once a case passes, a later change that breaks it is a regression you catch, not one you rediscover in production.

[↑ Back to top](#top)

## 6. Takeaways

- **You measure rates, not verdicts.** One green run on a non-deterministic agent can be luck. A **pass rate** over a few samples is the cheapest answer you can trust — the disciplined version of L11's "a single differing run proves nothing."
- **A flaky case is a finding.** `2/3` isn't noise to ignore; it means the case needs a looser check, more samples, or a redesign.
- **The eval set is a measurement instrument.** Running the *same* cases against two models turns "this one is weaker" into a number — and exposes headroom a single green run hides.
- **The ratchet is the payoff.** `compare(before, after)` flags pass→fail regressions before they ship. The same machinery answers "model A vs B" and "before vs after a prompt edit."
- **Next:** [`L1205_lab`](L1205_lab_empty.ipynb) — you run an eval set at `samples=K`, read pass rates, and flag regressions across two versions yourself. Then [`L1206_lecture`](L1206_lecture.ipynb) asks what all this *costs*.

[↑ Back to top](#top)